In [ ]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
import matplotlib.pyplot as plt

# Account / contract settings
ACCOUNT_SIZE = 50_000
RISK_PCT = 0.01
POINT_VALUE = 20  # NQ

# Risk rules
MIN_RISK_POINTS = 10
MAX_RISK_POINTS = 30
R_MULT = 3


In [ ]:
from pathlib import Path

BASE = Path(r"H:\1trade\QTxICT\data\cleaneddata")

def load_tf(path):
    df = pd.read_csv(path, parse_dates=["DateTime"])
    df = df.sort_values("DateTime").reset_index(drop=True)
    return df

data = {
    "NQ": {
        "1m": load_tf(BASE / "nq" / "1m_data.csv"),
        "5m": load_tf(BASE / "nq" / "5m_data.csv"),
        "15m": load_tf(BASE / "nq" / "15m_data.csv"),
        "1h": load_tf(BASE / "nq" / "1h_data.csv"),
        "1d": load_tf(BASE / "nq" / "1d_data.csv"),
        "1w": load_tf(BASE / "nq" / "1w_data.csv"),
    },
    "YM": {
        "1m": load_tf(BASE / "dow" / "1m_data.csv"),
        "5m": load_tf(BASE / "dow" / "5m_data.csv"),
        "15m": load_tf(BASE / "dow" / "15m_data.csv"),
        "1h": load_tf(BASE / "dow" / "1h_data.csv"),
        "1d": load_tf(BASE / "dow" / "1d_data.csv"),
        "1w": load_tf(BASE / "dow" / "1w_data.csv"),
    }
}

print("Loaded:")
for k in data:
    for tf in data[k]:
        print(k, tf, len(data[k][tf]))


Loaded:
NQ 1m 262126
NQ 5m 52428
NQ 15m 17476
NQ 1h 4412
NQ 1d 194
NQ 1w 39
YM 1m 187851
YM 5m 37637
YM 15m 12555
YM 1h 3141
YM 1d 138
YM 1w 28


In [ ]:
def fix_timezone(df):
    df = df.copy()
    if df["DateTime"].dt.tz is None:
        df["DateTime"] = (
            df["DateTime"]
            .dt.tz_localize("Etc/GMT-3")
            .dt.tz_convert("America/New_York")
        )
    return df

for sym in data:
    for tf in data[sym]:
        data[sym][tf] = fix_timezone(data[sym][tf])


In [ ]:
def in_entry_window_ny(ts: pd.Timestamp) -> bool:
    t = ts.tz_convert("America/New_York")
    return (
        (t.hour > 9 or (t.hour == 9 and t.minute >= 30))
        and
        (t.hour < 12)
    )


In [ ]:
@dataclass
class Event:
    ts: pd.Timestamp
    kind: str
    direction: str
    tf: str
    meta: dict


In [ ]:
def confirmed_pivots(series, left, right, mode):
    s = series.values
    out = np.zeros(len(s), dtype=bool)
    for i in range(left, len(s) - right):
        window = s[i-left:i+right+1]
        if mode == "high" and s[i] == window.max():
            out[i] = True
        if mode == "low" and s[i] == window.min():
            out[i] = True
    return out


In [ ]:
import pandas as pd
import numpy as np

def resample_ohlc(df_1m: pd.DataFrame, rule: str) -> pd.DataFrame:
    df = df_1m.set_index("DateTime")
    out = df.resample(rule, label="right", closed="right").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum",
        "TickVolume": "sum"
    }).dropna().reset_index()
    return out

# Use intersection timestamps for NQ/YM at 1m to keep comparisons clean
nq1m = data["NQ"]["1m"].copy()
ym1m = data["YM"]["1m"].copy()

# Rename
nq1m = nq1m.rename(columns={c: f"NQ_{c}" for c in ["Open","High","Low","Close","Volume","TickVolume"]})
ym1m = ym1m.rename(columns={c: f"YM_{c}" for c in ["Open","High","Low","Close","Volume","TickVolume"]})

bars_1m = pd.merge(nq1m, ym1m, on="DateTime", how="inner").sort_values("DateTime").reset_index(drop=True)

# Build per-asset 1m (needed for resampling)
nq_1m_raw = data["NQ"]["1m"].copy()
ym_1m_raw = data["YM"]["1m"].copy()

# Resample from 1m so all frames are consistent
frames = {}
for rule, name in [("5min","5m"), ("15min","15m"), ("90min","90m"), ("6H","6h"), ("1D","1d"), ("1W","1w")]:
    frames[f"NQ_{name}"] = resample_ohlc(nq_1m_raw, rule)
    frames[f"YM_{name}"] = resample_ohlc(ym_1m_raw, rule)

def merge_pair(tf_name: str) -> pd.DataFrame:
    a = frames[f"NQ_{tf_name}"][["DateTime","Open","High","Low","Close"]].rename(columns={c: f"NQ_{c}" for c in ["Open","High","Low","Close"]})
    b = frames[f"YM_{tf_name}"][["DateTime","Open","High","Low","Close"]].rename(columns={c: f"YM_{c}" for c in ["Open","High","Low","Close"]})
    return pd.merge(a, b, on="DateTime", how="inner").sort_values("DateTime").reset_index(drop=True)

bars_5m  = merge_pair("5m")
bars_15m = merge_pair("15m")
bars_90m = merge_pair("90m")
bars_6h  = merge_pair("6h")
bars_1d  = merge_pair("1d")
bars_1w  = merge_pair("1w")

bars_1m.head()


H:\temp\ipykernel_13440\954264942.py:6: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  out = df.resample(rule, label="right", closed="right").agg({
H:\temp\ipykernel_13440\954264942.py:6: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  out = df.resample(rule, label="right", closed="right").agg({


,DateTime,NQ_Open,NQ_High,NQ_Low,NQ_Close,NQ_Volume,NQ_TickVolume,YM_Open,YM_High,YM_Low,YM_Close,YM_Volume,YM_TickVolume
0,2025-01-01 17:01:00-05:00,21081.7,21092.1,21074.8,21092.1,0,139,42674.8,42679.8,42667.8,42679.8,0,28
1,2025-01-01 17:02:00-05:00,21092.7,21106.9,21092.7,21106.6,0,156,42681.8,42696.9,42681.8,42696.9,0,31
2,2025-01-01 17:03:00-05:00,21106.7,21124.4,21106.6,21118.9,0,202,42696.9,42715.8,42696.8,42713.4,0,31
3,2025-01-01 17:04:00-05:00,21118.5,21140.2,21117.5,21136.8,0,178,42713.8,42734.8,42713.8,42732.8,0,39
4,2025-01-01 17:05:00-05:00,21136.7,21140.8,21125.0,21130.4,0,207,42732.8,42736.8,42720.8,42720.8,0,38


In [ ]:
def build_narrative_states(bars_1w: pd.DataFrame, bars_1d: pd.DataFrame) -> pd.DataFrame:
    w = bars_1w[["DateTime","NQ_Open","NQ_High","NQ_Low","NQ_Close"]].copy()
    d = bars_1d[["DateTime","NQ_Open","NQ_High","NQ_Low","NQ_Close"]].copy()

    w["wk_dir"] = np.where(w["NQ_Close"] > w["NQ_Open"], "bull", "bear")
    d["dy_dir"] = np.where(d["NQ_Close"] > d["NQ_Open"], "bull", "bear")

    # Prior week EQ (midpoint) and range
    w["prev_wk_high"] = w["NQ_High"].shift(1)
    w["prev_wk_low"]  = w["NQ_Low"].shift(1)
    w["prev_wk_eq"]   = (w["prev_wk_high"] + w["prev_wk_low"]) / 2

    return w, d

wk_state, dy_state = build_narrative_states(bars_1w, bars_1d)

def latest_state_at(df: pd.DataFrame, t: pd.Timestamp):
    sub = df[df["DateTime"] <= t]
    if sub.empty:
        return None
    return sub.iloc[-1]

wk_state.tail(3), dy_state.tail(3)


(                    DateTime  NQ_Open  NQ_High   NQ_Low  NQ_Close wk_dir  \
 26 2025-07-06 00:00:00-04:00  22684.3  22906.4  22392.8   22793.1   bull   
 27 2025-07-13 00:00:00-04:00  22792.9  22935.4  22599.4   22712.5   bear   
 28 2025-07-20 00:00:00-04:00  22712.5  23168.4  22643.7   23103.9   bull   
 
     prev_wk_high  prev_wk_low  prev_wk_eq  
 26       22691.2      21538.3    22114.75  
 27       22906.4      22392.8    22649.60  
 28       22935.4      22599.4    22767.40  ,
                      DateTime  NQ_Open  NQ_High   NQ_Low  NQ_Close dy_dir
 164 2025-07-14 00:00:00-04:00  22701.1  22726.4  22641.6   22711.5   bull
 165 2025-07-15 00:00:00-04:00  22711.8  23014.2  22643.7   22935.2   bull
 166 2025-07-16 00:00:00-04:00  22935.0  23063.8  22820.0   22862.1   bear)

In [ ]:
def add_prev_period_levels(df: pd.DataFrame, period: str, prefix: str):
    """
    df indexed by DateTime with NQ_High/Low and YM_High/Low.
    period: 'M','W','D' etc using to_period.
    """
    out = df.copy()
    idx = out["DateTime"]

    # group key based on period
    key = idx.dt.to_period(period)

    # previous period high/low per asset
    out[f"NQ_prev_{prefix}_high"] = out.groupby(key)["NQ_High"].transform("max").shift(1)
    out[f"NQ_prev_{prefix}_low"]  = out.groupby(key)["NQ_Low"].transform("min").shift(1)
    out[f"YM_prev_{prefix}_high"] = out.groupby(key)["YM_High"].transform("max").shift(1)
    out[f"YM_prev_{prefix}_low"]  = out.groupby(key)["YM_Low"].transform("min").shift(1)

    return out

# Monthly draw (use 1D bars)
bars_1d_levels = add_prev_period_levels(bars_1d, "M", "month")
bars_1d_levels = add_prev_period_levels(bars_1d_levels, "W", "week")
bars_1d_levels = add_prev_period_levels(bars_1d_levels, "D", "day")  # previous day highs/lows from 1D bars is shift(1) anyway

# 6H draw (use 1H bars rolling prior 6H – already in your DC SSMT function, but we’ll keep levels too)
def add_prev_6h_levels_from_1h(bars_1h: pd.DataFrame):
    df = bars_1h.copy().set_index("DateTime")
    out = bars_1h.copy()
    out["NQ_prev_6h_high"] = df["NQ_High"].rolling("6H").max().shift(1).values
    out["NQ_prev_6h_low"]  = df["NQ_Low"].rolling("6H").min().shift(1).values
    out["YM_prev_6h_high"] = df["YM_High"].rolling("6H").max().shift(1).values
    out["YM_prev_6h_low"]  = df["YM_Low"].rolling("6H").min().shift(1).values
    return out

# 90m draw (use 90m bars; “previous 90m” is simply shift(1) range)
bars_90m_levels = bars_90m.copy()
bars_90m_levels["NQ_prev_90m_high"] = bars_90m_levels["NQ_High"].shift(1)
bars_90m_levels["NQ_prev_90m_low"]  = bars_90m_levels["NQ_Low"].shift(1)
bars_90m_levels["YM_prev_90m_high"] = bars_90m_levels["YM_High"].shift(1)
bars_90m_levels["YM_prev_90m_low"]  = bars_90m_levels["YM_Low"].shift(1)

# Micro draw (use 15m previous bar range)
bars_15m_levels = bars_15m.copy()
bars_15m_levels["NQ_prev_mc_high"] = bars_15m_levels["NQ_High"].shift(1)
bars_15m_levels["NQ_prev_mc_low"]  = bars_15m_levels["NQ_Low"].shift(1)
bars_15m_levels["YM_prev_mc_high"] = bars_15m_levels["YM_High"].shift(1)
bars_15m_levels["YM_prev_mc_low"]  = bars_15m_levels["YM_Low"].shift(1)

bars_1d_levels.head()


H:\temp\ipykernel_13440\3474475796.py:10: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = idx.dt.to_period(period)
H:\temp\ipykernel_13440\3474475796.py:10: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = idx.dt.to_period(period)
H:\temp\ipykernel_13440\3474475796.py:10: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  key = idx.dt.to_period(period)


,DateTime,NQ_Open,NQ_High,NQ_Low,NQ_Close,YM_Open,YM_High,YM_Low,YM_Close,NQ_prev_month_high,...,YM_prev_month_high,YM_prev_month_low,NQ_prev_week_high,NQ_prev_week_low,YM_prev_week_high,YM_prev_week_low,NQ_prev_day_high,NQ_prev_day_low,YM_prev_day_high,YM_prev_day_low
0,2025-01-02 00:00:00-05:00,21083.9,21197.1,20942.1,21188.6,42674.8,42791.8,42462.8,42778.8,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-03 00:00:00-05:00,21188.3,21311.1,20805.0,21047.1,42779.8,42998.8,42215.9,42476.8,21965.5,...,45058.9,41757.3,21380.1,20805.0,42998.8,42215.9,21197.1,20942.1,42791.8,42462.8
2,2025-01-04 00:00:00-05:00,21047.0,21380.1,21030.1,21313.0,42477.8,42827.9,42469.9,42745.8,21965.5,...,45058.9,41757.3,21380.1,20805.0,42998.8,42215.9,21311.1,20805.0,42998.8,42215.9
3,2025-01-06 00:00:00-05:00,21363.5,21379.2,21307.0,21367.9,42784.4,42790.8,42708.8,42759.8,21965.5,...,45058.9,41757.3,21380.1,20805.0,42998.8,42215.9,21380.1,21030.1,42827.9,42469.9
4,2025-01-07 00:00:00-05:00,21368.1,21724.0,21358.2,21536.7,42759.8,43165.9,42643.9,42722.8,21965.5,...,45058.9,41757.3,21724.0,20718.4,43165.9,41866.8,21379.2,21307.0,42790.8,42708.8


In [ ]:
# Build merged 1H bars (NQ + YM)
nq1h = data["NQ"]["1h"][["DateTime","Open","High","Low","Close"]].copy()
ym1h = data["YM"]["1h"][["DateTime","Open","High","Low","Close"]].copy()

nq1h = nq1h.rename(columns={c: f"NQ_{c}" for c in ["Open","High","Low","Close"]})
ym1h = ym1h.rename(columns={c: f"YM_{c}" for c in ["Open","High","Low","Close"]})

bars_1h = pd.merge(nq1h, ym1h, on="DateTime", how="inner").sort_values("DateTime").reset_index(drop=True)

bars_1h.head(), len(bars_1h)


(                   DateTime  NQ_Open  NQ_High   NQ_Low  NQ_Close  YM_Open  \
 0 2025-01-01 17:00:00-05:00  21083.9  21140.8  21065.1   21087.6  42674.8   
 1 2025-01-01 18:00:00-05:00  21087.5  21088.2  20942.1   21051.6  42618.8   
 2 2025-01-01 19:00:00-05:00  21051.6  21131.2  21051.6   21114.2  42605.8   
 3 2025-01-01 20:00:00-05:00  21113.7  21153.2  21107.2   21142.6  42669.8   
 4 2025-01-01 21:00:00-05:00  21142.7  21159.6  21115.0   21127.3  42738.8   
 
    YM_High   YM_Low  YM_Close  
 0  42736.8  42578.8   42617.8  
 1  42623.8  42462.8   42605.8  
 2  42694.8  42605.8   42669.8  
 3  42748.8  42663.8   42739.8  
 4  42755.8  42708.8   42718.8  ,
 3141)

In [ ]:
def add_prev_6h_levels_from_1h(bars_1h: pd.DataFrame) -> pd.DataFrame:
    """
    Adds prior-6-hour rolling high/low levels (shifted 1 bar to keep it causal).
    bars_1h must have DateTime, NQ_High/Low, YM_High/Low.
    """
    df = bars_1h.copy().set_index("DateTime")

    out = bars_1h.copy()
    out["NQ_prev_6h_high"] = df["NQ_High"].rolling("6h").max().shift(1).values
    out["NQ_prev_6h_low"]  = df["NQ_Low"].rolling("6h").min().shift(1).values
    out["YM_prev_6h_high"] = df["YM_High"].rolling("6h").max().shift(1).values
    out["YM_prev_6h_low"]  = df["YM_Low"].rolling("6h").min().shift(1).values
    return out


In [ ]:
# 90m levels (prev 90m bar high/low)
bars_90m_levels = bars_90m.copy()
bars_90m_levels["NQ_prev_90m_high"] = bars_90m_levels["NQ_High"].shift(1)
bars_90m_levels["NQ_prev_90m_low"]  = bars_90m_levels["NQ_Low"].shift(1)
bars_90m_levels["YM_prev_90m_high"] = bars_90m_levels["YM_High"].shift(1)
bars_90m_levels["YM_prev_90m_low"]  = bars_90m_levels["YM_Low"].shift(1)

# 15m levels (prev 15m bar high/low = "micro cycle" proxy)
bars_15m_levels = bars_15m.copy()
bars_15m_levels["NQ_prev_mc_high"] = bars_15m_levels["NQ_High"].shift(1)
bars_15m_levels["NQ_prev_mc_low"]  = bars_15m_levels["NQ_Low"].shift(1)
bars_15m_levels["YM_prev_mc_high"] = bars_15m_levels["YM_High"].shift(1)
bars_15m_levels["YM_prev_mc_low"]  = bars_15m_levels["YM_Low"].shift(1)


In [ ]:
def ssmt_vs_levels(df: pd.DataFrame, tf: str, hi_nq: str, hi_ym: str, lo_nq: str, lo_ym: str) -> pd.DataFrame:
    out = df[["DateTime","NQ_High","NQ_Low","YM_High","YM_Low"]].copy()

    out["bear_ssmt"] = (df["NQ_High"] > df[hi_nq]) & (df["YM_High"] <= df[hi_ym])
    out["bull_ssmt"] = (df["NQ_Low"]  < df[lo_nq]) & (df["YM_Low"]  >= df[lo_ym])

    out["tf"] = tf
    out["direction"] = np.where(out["bear_ssmt"], "bear", np.where(out["bull_ssmt"], "bull", None))
    out = out[out["direction"].notna()].copy()
    return out

# Monthly Cycle SSMT (1D vs prev month)
mc_ssmt = ssmt_vs_levels(
    bars_1d_levels, tf="Monthly(1D)",
    hi_nq="NQ_prev_month_high", hi_ym="YM_prev_month_high",
    lo_nq="NQ_prev_month_low",  lo_ym="YM_prev_month_low"
)

# Daily Cycle SSMT (1H vs prev 6h)
bars_1h_levels = add_prev_6h_levels_from_1h(bars_1h)
dc_ssmt = ssmt_vs_levels(
    bars_1h_levels, tf="Daily(6h on 1H)",
    hi_nq="NQ_prev_6h_high", hi_ym="YM_prev_6h_high",
    lo_nq="NQ_prev_6h_low",  lo_ym="YM_prev_6h_low"
)

# 90m Cycle SSMT (90m vs prev 90m)
c90_ssmt = ssmt_vs_levels(
    bars_90m_levels, tf="90m",
    hi_nq="NQ_prev_90m_high", hi_ym="YM_prev_90m_high",
    lo_nq="NQ_prev_90m_low",  lo_ym="YM_prev_90m_low"
)

# 15m SMTF (15m vs prior 15m “micro”)
smtf_15m = ssmt_vs_levels(
    bars_15m_levels, tf="15m(SMTF)",
    hi_nq="NQ_prev_mc_high", hi_ym="YM_prev_mc_high",
    lo_nq="NQ_prev_mc_low",  lo_ym="YM_prev_mc_low"
)

len(mc_ssmt), len(dc_ssmt), len(c90_ssmt), len(smtf_15m)


(2, 456, 426, 2724)

In [ ]:
def add_cisd_15m_proxy(df_15m: pd.DataFrame, body_ratio_min=0.6):
    df = df_15m.copy()
    body = (df["NQ_Close"] - df["NQ_Open"]).abs()
    rng  = (df["NQ_High"] - df["NQ_Low"]).replace(0, np.nan)
    df["body_ratio"] = (body / rng).fillna(0)

    # use prior bar high/low as “micro swing” proxy
    df["bull_cisd_15m"] = (df["NQ_Close"] > df["NQ_High"].shift(1)) & (df["body_ratio"] >= body_ratio_min)
    df["bear_cisd_15m"] = (df["NQ_Close"] < df["NQ_Low"].shift(1))  & (df["body_ratio"] >= body_ratio_min)

    # OB proxy zone
    df["ob_lo"] = np.where(df["bull_cisd_15m"], df["NQ_Open"], np.where(df["bear_cisd_15m"], df["NQ_Close"], np.nan))
    df["ob_hi"] = np.where(df["bull_cisd_15m"], df["NQ_Close"], np.where(df["bear_cisd_15m"], df["NQ_Open"], np.nan))
    df["ob_lo"] = np.minimum(df["ob_lo"], df["ob_hi"])
    df["ob_hi"] = np.maximum(df["ob_lo"], df["ob_hi"])

    return df

bars_15m_cisd = add_cisd_15m_proxy(bars_15m, body_ratio_min=0.6)
bars_15m_cisd[["DateTime","bull_cisd_15m","bear_cisd_15m","ob_lo","ob_hi"]].tail(10)


,DateTime,bull_cisd_15m,bear_cisd_15m,ob_lo,ob_hi
12548,2025-07-15 11:30:00-04:00,True,False,22973.0,23012.0
12549,2025-07-15 11:45:00-04:00,False,False,NaN,NaN
12550,2025-07-15 12:00:00-04:00,False,False,NaN,NaN
12551,2025-07-15 12:15:00-04:00,False,False,NaN,NaN
12552,2025-07-15 12:30:00-04:00,False,False,NaN,NaN
12553,2025-07-15 12:45:00-04:00,False,False,NaN,NaN
12554,2025-07-15 13:00:00-04:00,False,False,NaN,NaN
12555,2025-07-15 13:15:00-04:00,False,False,NaN,NaN
12556,2025-07-15 13:30:00-04:00,False,False,NaN,NaN
12557,2025-07-15 13:45:00-04:00,True,False,22961.5,22985.3


In [ ]:
# bars1 = 1m master used by the strategy
bars1 = bars_1m[[
    "DateTime",
    "NQ_Open","NQ_High","NQ_Low","NQ_Close",
    "YM_Open","YM_High","YM_Low","YM_Close"
]].copy()

# ---- 1m pivots on NQ for stop placement ----
PIV_L = 2
PIV_R = 2

bars1["pivH_c"] = confirmed_pivots(bars1["NQ_High"], PIV_L, PIV_R, "high")
bars1["pivL_c"] = confirmed_pivots(bars1["NQ_Low"],  PIV_L, PIV_R, "low")

bars1["last_pivH"] = np.where(bars1["pivH_c"], bars1["NQ_High"], np.nan)
bars1["last_pivL"] = np.where(bars1["pivL_c"], bars1["NQ_Low"],  np.nan)
bars1["last_pivH"] = bars1["last_pivH"].ffill()
bars1["last_pivL"] = bars1["last_pivL"].ffill()

# ---- 1m CISD proxy (displacement candle) ----
CISD_BODY_RATIO_MIN = 0.6

body = (bars1["NQ_Close"] - bars1["NQ_Open"]).abs()
rng  = (bars1["NQ_High"] - bars1["NQ_Low"]).replace(0, np.nan)
bars1["body_ratio"] = (body / rng).fillna(0)

# micro break proxy: close beyond prior bar high/low + displacement
bars1["bull_cisd"] = (bars1["NQ_Close"] > bars1["NQ_High"].shift(1)) & (bars1["body_ratio"] >= CISD_BODY_RATIO_MIN)
bars1["bear_cisd"] = (bars1["NQ_Close"] < bars1["NQ_Low"].shift(1))  & (bars1["body_ratio"] >= CISD_BODY_RATIO_MIN)

bars1.head()


,DateTime,NQ_Open,NQ_High,NQ_Low,NQ_Close,YM_Open,YM_High,YM_Low,YM_Close,pivH_c,pivL_c,last_pivH,last_pivL,body_ratio,bull_cisd,bear_cisd
0,2025-01-01 17:01:00-05:00,21081.7,21092.1,21074.8,21092.1,42674.8,42679.8,42667.8,42679.8,False,False,NaN,NaN,0.601156,False,False
1,2025-01-01 17:02:00-05:00,21092.7,21106.9,21092.7,21106.6,42681.8,42696.9,42681.8,42696.9,False,False,NaN,NaN,0.978873,True,False
2,2025-01-01 17:03:00-05:00,21106.7,21124.4,21106.6,21118.9,42696.9,42715.8,42696.8,42713.4,False,False,NaN,NaN,0.685393,True,False
3,2025-01-01 17:04:00-05:00,21118.5,21140.2,21117.5,21136.8,42713.8,42734.8,42713.8,42732.8,False,False,NaN,NaN,0.806167,True,False
4,2025-01-01 17:05:00-05:00,21136.7,21140.8,21125.0,21130.4,42732.8,42736.8,42720.8,42720.8,True,False,21140.8,NaN,0.398734,False,False


In [ ]:
def latest_event_before(df: pd.DataFrame, t: pd.Timestamp, direction: str | None = None):
    sub = df[df["DateTime"] <= t]
    if direction is not None:
        sub = sub[sub["direction"] == direction]
    if sub.empty:
        return None
    return sub.iloc[-1]

def latest_event_after(df: pd.DataFrame, start_ts: pd.Timestamp, end_ts: pd.Timestamp, direction: str | None = None):
    sub = df[(df["DateTime"] > start_ts) & (df["DateTime"] <= end_ts)]
    if direction is not None:
        sub = sub[sub["direction"] == direction]
    if sub.empty:
        return None
    return sub.iloc[-1]

def latest_signal_after(df: pd.DataFrame, start_ts: pd.Timestamp, end_ts: pd.Timestamp, signal_col: str):
    sub = df[(df["DateTime"] > start_ts) & (df["DateTime"] <= end_ts) & (df[signal_col] == True)]
    if sub.empty:
        return None
    return sub.iloc[-1]


In [ ]:
def get_latest_event(df: pd.DataFrame, t: pd.Timestamp):
    sub = df[df["DateTime"] <= t]
    if sub.empty:
        return None
    return sub.iloc[-1]

def narrative_dir_at(t: pd.Timestamp):
    w = wk_state[wk_state["DateTime"] <= t]
    d = dy_state[dy_state["DateTime"] <= t]
    if w.empty or d.empty:
        return None
    wdir = w.iloc[-1]["wk_dir"]
    ddir = d.iloc[-1]["dy_dir"]
    return wdir if wdir == ddir else None

def run_fearing_model(
    bars1: pd.DataFrame,
    mc_ssmt: pd.DataFrame,
    dc_ssmt: pd.DataFrame,
    c90_ssmt: pd.DataFrame,
    smtf_15m: pd.DataFrame,
    bars_15m_cisd: pd.DataFrame,
    R_MULT=2.0,
    MIN_RISK_POINTS=40,
    MAX_RISK_POINTS=60,
    REQUIRE_MONTHLY=False,   # <-- IMPORTANT: default off because you only have 2 monthly events
):
    trades = []
    open_trade = None

    for i in range(1, len(bars1)):
        t = bars1.loc[i, "DateTime"]

        # manage open trade
        if open_trade is not None:
            hi = bars1.loc[i, "NQ_High"]
            lo = bars1.loc[i, "NQ_Low"]
            if open_trade["direction"] == "long":
                if lo <= open_trade["stop"]:
                    open_trade.update(exit_ts=t, exit_price=open_trade["stop"], exit_reason="stop")
                    trades.append(open_trade); open_trade=None
                    continue
                if hi >= open_trade["target"]:
                    open_trade.update(exit_ts=t, exit_price=open_trade["target"], exit_reason="target")
                    trades.append(open_trade); open_trade=None
                    continue
            else:
                if hi >= open_trade["stop"]:
                    open_trade.update(exit_ts=t, exit_price=open_trade["stop"], exit_reason="stop")
                    trades.append(open_trade); open_trade=None
                    continue
                if lo <= open_trade["target"]:
                    open_trade.update(exit_ts=t, exit_price=open_trade["target"], exit_reason="target")
                    trades.append(open_trade); open_trade=None
                    continue

        if open_trade is not None:
            continue

        # session restriction
        if not in_entry_window_ny(t):
            continue

        # 1) narrative (weekly + daily agree)
        ndir = narrative_dir_at(t)
        if ndir is None:
            continue

        # 2) Monthly SSMT (optional gate)
        m = latest_event_before(mc_ssmt, t, direction=ndir)  # must match narrative if it exists
        if REQUIRE_MONTHLY and m is None:
            continue

        # chain start timestamp
        chain_start = m["DateTime"] if m is not None else (t - pd.Timedelta(days=90))

        # 3) Daily(6h) SSMT after chain_start, same direction as narrative
        d = latest_event_after(dc_ssmt, chain_start, t, direction=ndir)
        if d is None:
            continue

        # 4) 90m SSMT after daily, same direction
        n90 = latest_event_after(c90_ssmt, d["DateTime"], t, direction=ndir)
        if n90 is None:
            continue

        # 5) 15m SMTF after 90m, same direction
        f15 = latest_event_after(smtf_15m, n90["DateTime"], t, direction=ndir)
        if f15 is None:
            continue

        # 6) 15m CISD after SMTF (FIXED: must be a CISD bar, not just latest bar)
        sig_col = "bull_cisd_15m" if ndir == "bull" else "bear_cisd_15m"
        c15 = latest_signal_after(bars_15m_cisd, f15["DateTime"], t, sig_col)
        if c15 is None:
            continue

        # 7) 1m entry trigger (your 1m CISD)
        entry_price = float(bars1.loc[i, "NQ_Open"])
        lastH = bars1.loc[i-1, "last_pivH"]
        lastL = bars1.loc[i-1, "last_pivL"]
        if np.isnan(lastH) or np.isnan(lastL):
            continue

        if ndir == "bull" and bool(bars1.loc[i, "bull_cisd"]):
            stop = float(lastL)
            risk = entry_price - stop
            if risk < MIN_RISK_POINTS or risk > MAX_RISK_POINTS:
                continue
            target = entry_price + R_MULT * risk
            open_trade = dict(
    entry_ts=t,
    direction="long",
    entry_price=entry_price,
    stop=stop,
    target=float(target),

    # intent / sizing helpers
    intended_r=R_MULT,
    intended_reward_points=abs(target - entry_price),
    risk_points=abs(entry_price - stop),

    # filled on exit
    exit_ts=None,
    exit_price=None,
    exit_reason=None,

    # WHY we took it (independent permission)
    reason_ssmt_tf="most_recent",   # because we're using most recent SSMT across TFs
    ssmt_dir=sdir,
    ssmt_ts=s["DateTime"],
    ssmt_cycle_id=int(s["cycle_id"]),
    ssmt_sweeper=s["sweeper"],

    trigger="1m_CISD",
)


        if ndir == "bear" and bool(bars1.loc[i, "bear_cisd"]):
            stop = float(lastH)
            risk = stop - entry_price
            if risk < MIN_RISK_POINTS or risk > MAX_RISK_POINTS:
                continue
            target = entry_price - R_MULT * risk
            open_trade = dict(
    entry_ts=t,
    direction="short",
    entry_price=entry_price,
    stop=stop,
    target=float(target),

    # intent / sizing helpers
    intended_r=R_MULT,
    intended_reward_points=abs(entry_price - target),
    risk_points=abs(stop - entry_price),

    # filled on exit
    exit_ts=None,
    exit_price=None,
    exit_reason=None,

    # WHY we took it (independent permission)
    reason_ssmt_tf="most_recent",
    ssmt_dir=sdir,
    ssmt_ts=s["DateTime"],
    ssmt_cycle_id=int(s["cycle_id"]),
    ssmt_sweeper=s["sweeper"],

    trigger="1m_CISD",
)


    results = pd.DataFrame(trades)
    if results.empty:
        return results

    results["pnl_points"] = np.where(
        results["direction"] == "long",
        results["exit_price"] - results["entry_price"],
        results["entry_price"] - results["exit_price"]
    )
    results["risk_points"] = (results["entry_price"] - results["stop"]).abs()
    results["r_multiple"] = results["pnl_points"] / results["risk_points"].replace(0, np.nan)
    return results



In [ ]:
if results_fearing.empty:
    print("No trades produced by Fear.ing model filters (may be too strict).")
else:
    RISK_DOLLARS = ACCOUNT_SIZE * RISK_PCT

    r = results_fearing.copy()
    r["contracts"] = (RISK_DOLLARS / (r["risk_points"] * POINT_VALUE)).fillna(0).astype(int).clip(lower=1)
    r["pnl_dollars"] = r["pnl_points"] * POINT_VALUE * r["contracts"]

    r["month"] = r["entry_ts"].dt.to_period("M")
    monthly = r.groupby("month").agg(
        monthly_pnl=("pnl_dollars","sum"),
        trades=("pnl_dollars","size"),
        wins=("pnl_dollars", lambda x: int((x>0).sum())),
        losses=("pnl_dollars", lambda x: int((x<=0).sum()))
    )
    monthly["win_rate"] = monthly["wins"] / monthly["trades"]
    monthly["equity"] = ACCOUNT_SIZE + monthly["monthly_pnl"].cumsum()

    display(monthly)


No trades produced by Fear.ing model filters (may be too strict).


In [ ]:
print("bars1 cols:", [c for c in ["last_pivH","last_pivL","bull_cisd","bear_cisd"] if c in bars1.columns])
print("bull_cisd count:", int(bars1["bull_cisd"].sum()))
print("bear_cisd count:", int(bars1["bear_cisd"].sum()))
print("mc_ssmt/dc_ssmt/90m/15m sizes:", len(mc_ssmt), len(dc_ssmt), len(c90_ssmt), len(smtf_15m), len(bars_15m_cisd))
print("Narrative available bars:", sum(narrative_dir_at(t) is not None for t in bars1["DateTime"].iloc[::5000]))


bars1 cols: ['last_pivH', 'last_pivL', 'bull_cisd', 'bear_cisd']
bull_cisd count: 27378
bear_cisd count: 25019
mc_ssmt/dc_ssmt/90m/15m sizes: 2 456 426 2724 12558
Narrative available bars: 21


In [ ]:
import pandas as pd
import numpy as np

NY_TZ = "America/New_York"


# Use raw 1m bars for each asset (must be Open/High/Low/Close)
a_df = data["NQ"]["1m"][["DateTime", "Open", "High", "Low", "Close"]].copy()
b_df = data["YM"]["1m"][["DateTime", "Open", "High", "Low", "Close"]].copy()

# (optional) enforce sorting + timezone consistency
a_df = a_df.sort_values("DateTime").reset_index(drop=True)
b_df = b_df.sort_values("DateTime").reset_index(drop=True)

print(a_df.columns)
print(b_df.columns)
print(a_df["DateTime"].dt.tz, b_df["DateTime"].dt.tz)



# -----------------------------
# 1) Cycle builder (Quarterly Theory sessions + 90m quarters)
# -----------------------------
def add_cycle_ids(
    df: pd.DataFrame,
    dt_col="DateTime",
    tz="America/New_York",
    session_len_hours=6,
    quarter_minutes=90,
    micro_minutes=None,
):
    out = df.copy()
    out[dt_col] = pd.to_datetime(out[dt_col])

    # Ensure tz-aware NY
    if out[dt_col].dt.tz is None:
        out[dt_col] = out[dt_col].dt.tz_localize(tz)
    else:
        out[dt_col] = out[dt_col].dt.tz_convert(tz)

    ts = out[dt_col]

    # 6-hour blocks anchored at 18:00 NY:
    # shift time by +6h so the "day" starts at 18:00, then floor to 6H
    anchor_shift = pd.Timedelta(hours=6)
    session_start = (ts + anchor_shift).dt.floor(f"{session_len_hours}H") - anchor_shift
    session_end = session_start + pd.Timedelta(hours=session_len_hours)

    out["session_start"] = session_start
    out["session_end"] = session_end
    out["session_id"] = out["session_start"].astype("int64")

    mins_into = (ts - session_start).dt.total_seconds() / 60.0

    # 90m quarters inside the 6h session
    q_per_session = int((session_len_hours * 60) / quarter_minutes)  # should be 4
    quarter_idx = np.floor(mins_into / quarter_minutes).astype(int).clip(0, q_per_session - 1)

    out["quarter_idx"] = quarter_idx
    out["quarter_start"] = session_start + pd.to_timedelta(out["quarter_idx"] * quarter_minutes, unit="m")
    out["quarter_end"] = out["quarter_start"] + pd.Timedelta(minutes=quarter_minutes)
    out["quarter_id"] = out["quarter_start"].astype("int64")

    if micro_minutes is not None:
        micro_idx = np.floor(mins_into / micro_minutes).astype(int)
        out["cycle_start"] = session_start + pd.to_timedelta(micro_idx * micro_minutes, unit="m")
        out["cycle_end"] = out["cycle_start"] + pd.Timedelta(minutes=micro_minutes)
        out["cycle_id"] = out["cycle_start"].astype("int64")
        out["cycle_kind"] = f"micro_{micro_minutes}m"
    else:
        out["cycle_start"] = out["quarter_start"]
        out["cycle_end"] = out["quarter_end"]
        out["cycle_id"] = out["quarter_id"]
        out["cycle_kind"] = f"{quarter_minutes}m"

    return out



# -----------------------------
# 4) "Most recent valid SSMT" at time t
# -----------------------------
def most_recent_valid_ssmt(events: pd.DataFrame, t: pd.Timestamp, dt_col="DateTime"):
    if events.empty:
        return None
    t = pd.to_datetime(t)
    if t.tzinfo is None and events[dt_col].dt.tz is not None:
        t = t.tz_localize(events[dt_col].dt.tz)

    valid = events[(events[dt_col] <= t) & (events["cycle_end"] >= t)]
    if valid.empty:
        return None
    return valid.iloc[-1]


Index(['DateTime', 'Open', 'High', 'Low', 'Close'], dtype='object')
Index(['DateTime', 'Open', 'High', 'Low', 'Close'], dtype='object')
America/New_York America/New_York


In [ ]:
# a_df and b_df must have: DateTime, Open, High, Low, Close (1m bars)
a1 = add_cycle_ids(a_df, quarter_minutes=90, micro_minutes=None)   # 90m cycles
b1 = add_cycle_ids(b_df, quarter_minutes=90, micro_minutes=None)

a1 = add_prev_cycle_levels(a1)
b1 = add_prev_cycle_levels(b1)

ssmt_90m = detect_ssmt_events(a1, b1, require_reclaim=True)

# At some timestamp t:
t = a1["DateTime"].iloc[-1]
active = most_recent_valid_ssmt(ssmt_90m, t)
print(active)


H:\temp\ipykernel_13440\3488818410.py:46: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  session_start = (ts + anchor_shift).dt.floor(f"{session_len_hours}H") - anchor_shift
H:\temp\ipykernel_13440\3488818410.py:46: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  session_start = (ts + anchor_shift).dt.floor(f"{session_len_hours}H") - anchor_shift


None


In [ ]:
# 15m "micro" SSMT (proxy)
a15 = add_cycle_ids(a_df, quarter_minutes=90, micro_minutes=15)
b15 = add_cycle_ids(b_df, quarter_minutes=90, micro_minutes=15)
a15 = add_prev_cycle_levels(a15)
b15 = add_prev_cycle_levels(b15)
ssmt_15m = detect_ssmt_events(a15, b15, require_reclaim=False)

# 90m SSMT
a90 = add_cycle_ids(a_df, quarter_minutes=90, micro_minutes=None)
b90 = add_cycle_ids(b_df, quarter_minutes=90, micro_minutes=None)
a90 = add_prev_cycle_levels(a90)
b90 = add_prev_cycle_levels(b90)
ssmt_90m = detect_ssmt_events(a90, b90, require_reclaim=False)

# 6h SSMT (session cycle) — treat each 6h as a cycle (micro_minutes=360)
a6 = add_cycle_ids(a_df, quarter_minutes=90, micro_minutes=360)
b6 = add_cycle_ids(b_df, quarter_minutes=90, micro_minutes=360)
a6 = add_prev_cycle_levels(a6)
b6 = add_prev_cycle_levels(b6)
ssmt_6h = detect_ssmt_events(a6, b6, require_reclaim=False)

print(len(ssmt_15m), len(ssmt_90m), len(ssmt_6h))

def most_recent_ssmt_across(t: pd.Timestamp, *event_dfs: pd.DataFrame):
    pool = []
    for df in event_dfs:
        if df is None or df.empty:
            continue
        sub = df[df["DateTime"] <= t]
        if not sub.empty:
            pool.append(sub.iloc[-1])
    if not pool:
        return None
    # pick latest timestamp among the latest-per-tf
    return max(pool, key=lambda r: r["DateTime"])

def backtest_ssmt_cisd(
    bars1: pd.DataFrame,
    ssmt_15m: pd.DataFrame,
    ssmt_90m: pd.DataFrame,
    ssmt_6h: pd.DataFrame,
    bars_15m_cisd: pd.DataFrame,   # optional confirm
    require_15m_cisd=False,        # start False
    R_MULT=2.0,
    MIN_RISK_POINTS=40,
    MAX_RISK_POINTS=60,
):
    trades = []
    open_trade = None

    for i in range(1, len(bars1)):
        t = bars1.loc[i, "DateTime"]

        # manage open trade
        if open_trade is not None:
            hi = bars1.loc[i, "NQ_High"]
            lo = bars1.loc[i, "NQ_Low"]
            if open_trade["direction"] == "long":
                if lo <= open_trade["stop"]:
                    open_trade.update(exit_ts=t, exit_price=open_trade["stop"], exit_reason="stop")
                    trades.append(open_trade); open_trade = None
                    continue
                if hi >= open_trade["target"]:
                    open_trade.update(exit_ts=t, exit_price=open_trade["target"], exit_reason="target")
                    trades.append(open_trade); open_trade = None
                    continue
            else:
                if hi >= open_trade["stop"]:
                    open_trade.update(exit_ts=t, exit_price=open_trade["stop"], exit_reason="stop")
                    trades.append(open_trade); open_trade = None
                    continue
                if lo <= open_trade["target"]:
                    open_trade.update(exit_ts=t, exit_price=open_trade["target"], exit_reason="target")
                    trades.append(open_trade); open_trade = None
                    continue

        if open_trade is not None:
            continue

        # entry window
        if not in_entry_window_ny(t):
            continue

        # 1) SSMT permission (most recent wins)
        s = most_recent_ssmt_across(t, ssmt_15m, ssmt_90m, ssmt_6h)
        if s is None:
            continue

        sdir = s["dir"]  # 'bull' or 'bear'

        # optional 15m CISD confirm AFTER the SSMT print
        if require_15m_cisd:
            sig_col = "bull_cisd_15m" if sdir == "bull" else "bear_cisd_15m"
            c15 = latest_signal_after(bars_15m_cisd, s["DateTime"], t, sig_col)
            if c15 is None:
                continue

        # 2) 1m CISD trigger
        if sdir == "bull" and not bool(bars1.loc[i, "bull_cisd"]):
            continue
        if sdir == "bear" and not bool(bars1.loc[i, "bear_cisd"]):
            continue

        entry_price = float(bars1.loc[i, "NQ_Open"])
        lastH = bars1.loc[i-1, "last_pivH"]
        lastL = bars1.loc[i-1, "last_pivL"]
        if np.isnan(lastH) or np.isnan(lastL):
            continue

        if sdir == "bull":
            stop = float(lastL)
            risk = entry_price - stop
            if risk < MIN_RISK_POINTS or risk > MAX_RISK_POINTS:
                continue
            target = entry_price + R_MULT * risk
            open_trade = dict(
    entry_ts=t,
    direction="long",
    entry_price=entry_price,
    stop=stop,
    target=float(target),

    # intent / sizing helpers
    intended_r=R_MULT,
    intended_reward_points=abs(target - entry_price),
    risk_points=abs(entry_price - stop),

    # filled on exit
    exit_ts=None,
    exit_price=None,
    exit_reason=None,

    # WHY we took it (independent permission)
    reason_ssmt_tf="most_recent",   # because we're using most recent SSMT across TFs
    ssmt_dir=sdir,
    ssmt_ts=s["DateTime"],
    ssmt_cycle_id=int(s["cycle_id"]),
    ssmt_sweeper=s["sweeper"],

    trigger="1m_CISD",
)

        else:
            stop = float(lastH)
            risk = stop - entry_price
            if risk < MIN_RISK_POINTS or risk > MAX_RISK_POINTS:
                continue
            target = entry_price - R_MULT * risk
            open_trade = dict(
    entry_ts=t,
    direction="short",
    entry_price=entry_price,
    stop=stop,
    target=float(target),

    # intent / sizing helpers
    intended_r=R_MULT,
    intended_reward_points=abs(entry_price - target),
    risk_points=abs(stop - entry_price),

    # filled on exit
    exit_ts=None,
    exit_price=None,
    exit_reason=None,

    # WHY we took it (independent permission)
    reason_ssmt_tf="most_recent",
    ssmt_dir=sdir,
    ssmt_ts=s["DateTime"],
    ssmt_cycle_id=int(s["cycle_id"]),
    ssmt_sweeper=s["sweeper"],

    trigger="1m_CISD",
)


    results = pd.DataFrame(trades)
    if results.empty:
        return results

    results["pnl_points"] = np.where(
        results["direction"] == "long",
        results["exit_price"] - results["entry_price"],
        results["entry_price"] - results["exit_price"]
    )
    results["risk_points"] = (results["entry_price"] - results["stop"]).abs()
    results["r_multiple"] = results["pnl_points"] / results["risk_points"].replace(0, np.nan)
    return results


H:\temp\ipykernel_13440\3488818410.py:46: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  session_start = (ts + anchor_shift).dt.floor(f"{session_len_hours}H") - anchor_shift
H:\temp\ipykernel_13440\3488818410.py:46: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  session_start = (ts + anchor_shift).dt.floor(f"{session_len_hours}H") - anchor_shift
H:\temp\ipykernel_13440\3488818410.py:46: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  session_start = (ts + anchor_shift).dt.floor(f"{session_len_hours}H") - anchor_shift
H:\temp\ipykernel_13440\3488818410.py:46: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  session_start = (ts + anchor_shift).dt.floor(f"{session_len_hours}H") - anchor_shift
H:\temp\ipykernel_13440\3488818410.py:46: FutureWarning: 'H' is deprecated and will be remov

13929 2898 808


In [ ]:
results = backtest_ssmt_cisd(
    bars1,
    ssmt_15m=ssmt_15m,
    ssmt_90m=ssmt_90m,
    ssmt_6h=ssmt_6h,
    bars_15m_cisd=bars_15m_cisd,
    require_15m_cisd=False,  # start off
)

print("trades:", len(results))
results.head()


trades: 144


,entry_ts,direction,entry_price,stop,target,intended_r,intended_reward_points,risk_points,exit_ts,exit_price,exit_reason,reason_ssmt_tf,ssmt_dir,ssmt_ts,ssmt_cycle_id,ssmt_sweeper,trigger,pnl_points,r_multiple
0,2025-01-02 10:51:00-05:00,long,21164.2,21113.4,21265.8,2.0,101.6,50.8,2025-01-02 10:54:00-05:00,21113.4,stop,most_recent,bull,2025-01-02 10:49:00-05:00,1735815600000000000,B,1m_CISD,-50.8,-1.0
1,2025-01-02 11:20:00-05:00,short,21118.2,21163.0,21028.6,2.0,89.6,44.8,2025-01-02 11:27:00-05:00,21028.6,target,most_recent,bear,2025-01-02 11:13:00-05:00,1735833600000000000,B,1m_CISD,89.6,2.0
2,2025-01-02 11:51:00-05:00,long,20848.9,20805.0,20936.7,2.0,87.8,43.9,2025-01-02 13:25:00-05:00,20936.7,target,most_recent,bull,2025-01-02 11:48:00-05:00,1735836300000000000,A,1m_CISD,87.8,2.0
3,2025-01-07 09:52:00-05:00,long,21288.7,21230.6,21404.9,2.0,116.2,58.1,2025-01-07 11:46:00-05:00,21230.6,stop,most_recent,bull,2025-01-07 09:46:00-05:00,1736261100000000000,B,1m_CISD,-58.1,-1.0
4,2025-01-07 11:59:00-05:00,long,21263.0,21218.0,21353.0,2.0,90.0,45.0,2025-01-07 14:09:00-05:00,21218.0,stop,most_recent,bull,2025-01-07 11:55:00-05:00,1736268300000000000,B,1m_CISD,-45.0,-1.0


In [ ]:
results = backtest_ssmt_cisd(
    bars1,
    ssmt_15m=ssmt_15m,
    ssmt_90m=ssmt_90m,
    ssmt_6h=ssmt_6h,
    bars_15m_cisd=bars_15m_cisd,
    require_15m_cisd=False,
    R_MULT=R_MULT,
    MIN_RISK_POINTS=MIN_RISK_POINTS,
    MAX_RISK_POINTS=MAX_RISK_POINTS,
)

if results.empty:
    print("No trades")
else:
    results["pnl_points"] = np.where(
        results["direction"] == "long",
        results["exit_price"] - results["entry_price"],
        results["entry_price"] - results["exit_price"]
    )
    results["r_multiple"] = results["pnl_points"] / results["risk_points"].replace(0, np.nan)

print("trades:", len(results))
results.head()


trades: 629


,entry_ts,direction,entry_price,stop,target,intended_r,intended_reward_points,risk_points,exit_ts,exit_price,exit_reason,reason_ssmt_tf,ssmt_dir,ssmt_ts,ssmt_cycle_id,ssmt_sweeper,trigger,pnl_points,r_multiple
0,2025-01-02 09:36:00-05:00,long,21070.3,21051.7,21126.1,3,55.8,18.6,2025-01-02 09:41:00-05:00,21051.7,stop,most_recent,bull,2025-01-02 09:30:00-05:00,1735828200000000000,B,1m_CISD,-18.6,-1.0
1,2025-01-02 09:48:00-05:00,long,21051.6,21034.6,21102.6,3,51.0,17.0,2025-01-02 09:49:00-05:00,21102.6,target,most_recent,bull,2025-01-02 09:30:00-05:00,1735828200000000000,B,1m_CISD,51.0,3.0
2,2025-01-02 10:09:00-05:00,long,21027.3,21007.6,21086.4,3,59.1,19.7,2025-01-02 10:19:00-05:00,21086.4,target,most_recent,bull,2025-01-02 10:02:00-05:00,1735830000000000000,B,1m_CISD,59.1,3.0
3,2025-01-02 10:43:00-05:00,short,21133.1,21145.9,21094.7,3,38.4,12.8,2025-01-02 10:48:00-05:00,21145.9,stop,most_recent,bear,2025-01-02 10:34:00-05:00,1735831800000000000,A,1m_CISD,-12.8,-1.0
4,2025-01-02 10:58:00-05:00,long,21133.7,21110.0,21204.8,3,71.1,23.7,2025-01-02 11:08:00-05:00,21110.0,stop,most_recent,bull,2025-01-02 10:49:00-05:00,1735815600000000000,B,1m_CISD,-23.7,-1.0


In [ ]:
if results.empty:
    print("No trades")
else:
    wins = (results["pnl_points"] > 0).mean()
    avg_R = results["r_multiple"].mean()
    med_R = results["r_multiple"].median()

    avg_risk = results["risk_points"].mean()
    avg_target_pts = results["intended_reward_points"].mean()
    avg_realized_pts = results["pnl_points"].mean()

    print(f"Trades: {len(results)}")
    print(f"Win rate: {wins:.2%}")
    print(f"Avg R: {avg_R:.3f} | Median R: {med_R:.3f}")
    print(f"Avg risk (pts): {avg_risk:.2f}")
    print(f"Avg points sought (target pts): {avg_target_pts:.2f}")
    print(f"Avg points realized (pnl pts): {avg_realized_pts:.2f}")

    results[[
        "entry_ts","direction",
        "ssmt_ts","ssmt_dir","ssmt_sweeper",
        "risk_points","intended_reward_points",
        "pnl_points","r_multiple","exit_reason"
    ]].head(25)


Trades: 629
Win rate: 51.67%
Avg R: 1.067 | Median R: 3.000
Avg risk (pts): 17.39
Avg points sought (target pts): 52.18
Avg points realized (pnl pts): 17.47


In [ ]:
if results.empty:
    print("No trades")
else:
    breakdown = (
        results
        .groupby(["ssmt_dir","ssmt_sweeper","exit_reason"])
        .agg(
            trades=("entry_ts","count"),
            win_rate=("pnl_points", lambda x: (x>0).mean()),
            avg_R=("r_multiple","mean"),
            avg_pts=("pnl_points","mean"),
        )
        .sort_values("trades", ascending=False)
    )
    display(breakdown)


trades  win_rate  avg_R    avg_pts
ssmt_dir ssmt_sweeper exit_reason                                    
bull     A            target           95       1.0    3.0  50.946316
bear     A            target           84       1.0    3.0  47.332143
bull     B            stop             84       0.0   -1.0 -17.783333
                      target           81       1.0    3.0  53.277778
         A            stop             79       0.0   -1.0 -18.955696
bear     B            stop             73       0.0   -1.0 -18.387671
         A            stop             68       0.0   -1.0 -16.550000
         B            target           65       1.0    3.0  51.009231

In [ ]:
# --- add $ sizing columns if missing ---
RISK_DOLLARS = ACCOUNT_SIZE * RISK_PCT

if "pnl_points" not in results.columns:
    results["pnl_points"] = np.where(
        results["direction"] == "long",
        results["exit_price"] - results["entry_price"],
        results["entry_price"] - results["exit_price"]
    )

if "risk_points" not in results.columns:
    results["risk_points"] = (results["entry_price"] - results["stop"]).abs()

# contracts based on fixed $ risk per trade
results["contracts"] = (RISK_DOLLARS / (results["risk_points"] * POINT_VALUE)).fillna(0).astype(int)
results["contracts"] = results["contracts"].clip(lower=1)

# dollars pnl
results["pnl_dollars"] = results["pnl_points"] * POINT_VALUE * results["contracts"]

# --- monthly table ---
results["month"] = results["entry_ts"].dt.tz_convert("America/New_York").dt.to_period("M")

monthly = results.groupby("month").agg(
    monthly_pnl=("pnl_dollars","sum"),
    trades=("entry_ts","count"),
    wins=("pnl_dollars", lambda x: int((x > 0).sum())),
    losses=("pnl_dollars", lambda x: int((x <= 0).sum())),
)

monthly["win_rate"] = monthly["wins"] / monthly["trades"]
monthly["equity"] = ACCOUNT_SIZE + monthly["monthly_pnl"].cumsum()

print("Total trades in results:", len(results))
print("Sum of monthly trades:", int(monthly["trades"].sum()))
display(monthly)



Total trades in results: 629
Sum of monthly trades: 629


H:\temp\ipykernel_13440\433088543.py:22: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  results["month"] = results["entry_ts"].dt.tz_convert("America/New_York").dt.to_period("M")


,monthly_pnl,trades,wins,losses,win_rate,equity
month,,,,,,
2025-01,32304.0,89,41,48,0.460674,82304.0
2025-02,31736.0,85,44,41,0.517647,114040.0
2025-03,33080.0,112,50,62,0.446429,147120.0
2025-04,75536.0,116,74,42,0.637931,222656.0
2025-05,38768.0,104,52,52,0.500000,261424.0
2025-06,34896.0,83,43,40,0.518072,296320.0
2025-07,17312.0,40,21,19,0.525000,313632.0


In [ ]:
# --- Trade viewer table ---

# pick a few helpful columns if they exist
cols = [
    "entry_ts","exit_ts","direction",
    "entry_price","stop","target",
    "risk_points","pnl_points","r_multiple","exit_reason",
    "ssmt_ts","ssmt_dir","ssmt_sweeper",
    "contracts","pnl_dollars",
]

view = results.copy()

# ensure pnl_points / r_multiple exist
if "pnl_points" not in view.columns:
    view["pnl_points"] = np.where(
        view["direction"] == "long",
        view["exit_price"] - view["entry_price"],
        view["entry_price"] - view["exit_price"]
    )
if "risk_points" not in view.columns:
    view["risk_points"] = (view["entry_price"] - view["stop"]).abs()
if "r_multiple" not in view.columns:
    view["r_multiple"] = view["pnl_points"] / view["risk_points"].replace(0, np.nan)

# keep only columns that exist
cols = [c for c in cols if c in view.columns]

# optional: filter examples
# view = view[view["direction"] == "long"]
# view = view[view["exit_reason"] == "target"]

# sort newest first
view = view.sort_values("entry_ts", ascending=False)

display(view[cols].head(50))


NameError: name 'results' is not defined

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

NY_TZ  = "America/New_York"

# IMPORTANT:
# Set this to the timezone your raw timestamps are ACTUALLY in.
# If you believe they're UTC+3, use "Etc/GMT-3" (yes, the sign is reversed in Etc/GMT zones).
SRC_TZ = "Etc/GMT-3"   # UTC+3
# If they're UTC, use: SRC_TZ = "UTC"

def _to_ny(ts, src_tz=SRC_TZ, ny_tz=NY_TZ) -> pd.Timestamp:
    ts = pd.to_datetime(ts)
    if ts.tzinfo is None:
        ts = ts.tz_localize(src_tz)
    else:
        ts = ts.tz_convert(src_tz)  # normalize if it came in with some tz
    return ts.tz_convert(ny_tz)

def _series_to_ny(s: pd.Series, src_tz=SRC_TZ, ny_tz=NY_TZ) -> pd.Series:
    s = pd.to_datetime(s)
    # If pandas parsed a tz-aware series, convert; otherwise localize then convert.
    if getattr(s.dt, "tz", None) is None:
        s = s.dt.tz_localize(src_tz)
    else:
        s = s.dt.tz_convert(src_tz)
    return s.dt.tz_convert(ny_tz)

def plot_trade(trade_row, bars1, minutes_before=60, minutes_after=180,
               price_col="NQ_Close", datetime_col="DateTime"):

    df_all = bars1.copy()
    df_all[datetime_col] = _series_to_ny(df_all[datetime_col])

    # ---- FORCE trade timestamps to NY ----
    t0 = _to_ny(trade_row["entry_ts"])
    exit_ts = trade_row.get("exit_ts", pd.NaT)
    t1 = _to_ny(exit_ts) if pd.notna(exit_ts) else (t0 + pd.Timedelta(minutes=minutes_after))

    start = t0 - pd.Timedelta(minutes=minutes_before)
    end   = t1 + pd.Timedelta(minutes=minutes_after)

    df = df_all[(df_all[datetime_col] >= start) & (df_all[datetime_col] <= end)].copy()
    if df.empty:
        print("No bars found for that window.")
        print("start:", start, "| end:", end)
        print("bars1 DateTime min/max:", df_all[datetime_col].min(), df_all[datetime_col].max())
        return

    x = df[datetime_col]
    price = df[price_col]

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(x, price)

    # ---- Levels ----
    entry = float(trade_row["entry_price"])
    stop  = float(trade_row["stop"])
    tp    = float(trade_row["target"])

    ax.axhline(entry, linestyle="--")
    ax.axhline(stop,  linestyle="--")
    ax.axhline(tp,    linestyle="--")

    # ---- Vertical markers ----
    ax.axvline(t0, linestyle="--")
    if pd.notna(exit_ts):
        ax.axvline(t1, linestyle="--")

    # ---- Label levels (right edge) ----
    label_x = x.iloc[-1]
    ax.text(label_x, entry, f" Entry: {entry:.2f}", va="center", ha="left")
    ax.text(label_x, stop,  f" Stop:  {stop:.2f}",  va="center", ha="left")
    ax.text(label_x, tp,    f" TP:    {tp:.2f}",    va="center", ha="left")

    # ---- Title ----
    title = (
        f"{str(trade_row.get('direction','')).upper()} | "
        f"{t0.strftime('%Y-%m-%d %H:%M')} NY | "
        f"Entry {entry:.2f} | Stop {stop:.2f} | TP {tp:.2f} | "
        f"Exit: {trade_row.get('exit_reason','')}"
    )
    ax.set_title(title)

    # ---- X-axis formatting ----
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d\n%H:%M", tz=x.dt.tz))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())

    ax.set_xlabel("Date / Time (New York)")
    ax.set_ylabel("NQ Price/penis")

    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ----------------------------
# Choose a trade to plot
# ----------------------------

# Option A (recommended): you already have a trades dataframe
# trade_to_plot = trades.iloc[600]

# Option B: if your trades df is named something else, set it here:
# trade_to_plot = view.iloc[600]   # only if you actually have `view` defined

# Option C: pass a row directly (example):
# trade_to_plot = some_df.loc[some_index]

# Then:
# plot_trade(trade_to_plot, bars1)
